# Paper Analysis & Figure Generation
Generate all figures, tables, and analysis for CYBERPUNK@DravidianLangTech 2026 paper

## 1. Setup & Load Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import LinearSVC
import torch
from PIL import Image
import open_clip
import cv2
from tqdm import tqdm

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded")

In [ ]:
# Update these paths to match your system
ROOT = "/kaggle/input/shared-poli-dataset/dataset"  # UPDATE THIS
TAMIL_IMG_DIR = os.path.join(ROOT, "Train_images(Tamil)")
MAL_IMG_DIR   = os.path.join(ROOT, "Train_images(Malayam)")
TAMIL_XLSX = os.path.join(ROOT, "Tamil_Train_labels.xlsx")
MAL_XLSX   = os.path.join(ROOT, "Malayalam_Train_label.xlsx")

# Output directory for figures
OUTPUT_DIR = "paper_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Output directory: {OUTPUT_DIR}")

## 2. Load and Prepare Dataset

In [ ]:
# Load data (reuse your existing loading code)
t = pd.read_excel(TAMIL_XLSX)
m = pd.read_excel(MAL_XLSX)

t = t.rename(columns={"Level1":"Level 1", "Level2":"Level 2"})
t["lang"] = "tamil"
m["lang"] = "malayalam"

def tamil_candidates(row):
    cands = []
    if "Image_name" in row and pd.notna(row["Image_name"]):
        cands.append(str(row["Image_name"]).strip())
    if "Image_id" in row and pd.notna(row["Image_id"]):
        cands.append(f"{int(row['Image_id'])}.jpg")
        cands.append(f"{int(row['Image_id']):03d}.jpg")
    return list(dict.fromkeys(cands))

def mal_candidates(row):
    cands = []
    if "meme_id" in row and pd.notna(row["meme_id"]):
        cands.append(f"{int(row['meme_id'])}.jpg")
        cands.append(f"{int(row['meme_id']):03d}.jpg")
    return list(dict.fromkeys(cands))

def resolve_path(img_dir, candidates):
    for fn in candidates:
        p = os.path.join(img_dir, fn)
        if os.path.exists(p):
            return p
    return None

t["image_path"] = t.apply(lambda r: resolve_path(TAMIL_IMG_DIR, tamil_candidates(r)), axis=1)
m["image_path"] = m.apply(lambda r: resolve_path(MAL_IMG_DIR, mal_candidates(r)), axis=1)

print(f"Tamil images: {len(t)}, Malayalam images: {len(m)}")

In [ ]:
# Map to internal labels
def map_internal(lang, level1, level2):
    if lang == "tamil":
        stance = "SUPPORT" if level1 == "Support/Praise" else "TROLL"
        if level2 in ["Support for person", "Troll/Oppose Against Person"]:
            target = "PERSON"
        elif level2 in ["Support for party", "Troll/Oppose Against Party"]:
            target = "PARTY"
        else:
            target = "UNKNOWN"
    else:
        stance = "SUPPORT" if str(level1).strip().upper() == "SUPPORT" else "TROLL"
        l2 = str(level2).strip()
        if l2 in ["Against individual person", "Support for individual person"]:
            target = "PERSON"
        elif l2 in ["Against party", "Support for party"]:
            target = "PARTY"
        elif l2 == "Intersection":
            target = "INTERSECTION"
        else:
            target = "UNKNOWN"
    return stance, target

def add_internal(df):
    stances, targets = [], []
    for _, r in df.iterrows():
        stance, target = map_internal(r["lang"], r["Level 1"], r["Level 2"])
        stances.append(stance)
        targets.append(target)
    df["stance_internal"] = stances
    df["target_internal"] = targets
    return df

t = add_internal(t)
m = add_internal(m)
df = pd.concat([t, m], ignore_index=True)
df = df[df["image_path"].notna()].reset_index(drop=True)

print(f"Total samples: {len(df)}")
print(f"Tamil: {len(df[df['lang']=='tamil'])}, Malayalam: {len(df[df['lang']=='malayalam'])}")

## 3. Dataset Statistics Table (for paper)

In [ ]:
# TABLE 1: Dataset Statistics
def dataset_stats():
    stats = []
    
    for lang in ['tamil', 'malayalam']:
        lang_df = df[df['lang'] == lang]
        
        # Stance distribution
        support = len(lang_df[lang_df['stance_internal'] == 'SUPPORT'])
        troll = len(lang_df[lang_df['stance_internal'] == 'TROLL'])
        
        # Target distribution
        person = len(lang_df[lang_df['target_internal'] == 'PERSON'])
        party = len(lang_df[lang_df['target_internal'] == 'PARTY'])
        intersection = len(lang_df[lang_df['target_internal'] == 'INTERSECTION'])
        
        stats.append({
            'Language': lang.title(),
            'Total': len(lang_df),
            'Support': support,
            'Troll': troll,
            'Person': person,
            'Party': party,
            'Intersection': intersection if intersection > 0 else '-'
        })
    
    stats_df = pd.DataFrame(stats)
    print("\n=== TABLE 1: Dataset Statistics ===")
    print(stats_df.to_string(index=False))
    print("\nLaTeX format:")
    print(stats_df.to_latex(index=False))
    
    return stats_df

stats_df = dataset_stats()

## 4. Extract Features (CLIP + Face + Logo)

In [ ]:
# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="laion2b_s32b_b82k"
)
model_clip = model_clip.to(device).eval()

@torch.inference_mode()
def embed_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_t = preprocess(img).unsqueeze(0).to(device)
    emb = model_clip.encode_image(img_t)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze(0).cpu().numpy()

print(f"✅ CLIP model loaded on {device}")

In [ ]:
# Extract CLIP embeddings
print("Extracting CLIP embeddings...")
embs = []
for p in tqdm(df["image_path"].tolist(), desc="CLIP"):
    embs.append(embed_image(p))
img_embs = np.vstack(embs)
print(f"CLIP embeddings shape: {img_embs.shape}")

In [ ]:
# Face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def face_stats(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return 0, 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    num = len(faces)
    if num == 0:
        return 0, 0.0
    h, w = gray.shape
    areas = [(fw * fh) / (w * h) for (x, y, fw, fh) in faces]
    return num, float(max(areas))

print("Extracting face features...")
face_results = [face_stats(p) for p in tqdm(df["image_path"].tolist(), desc="Faces")]
df["num_faces"], df["max_face_area"] = zip(*face_results)
face_feats = df[["num_faces", "max_face_area"]].to_numpy().astype(np.float32)
print(f"Face features shape: {face_feats.shape}")

## 4.3 Logo Similarity Features

In [ ]:
# STEP 2 — Load and embed ALL logos (exactly as in st2-multimodal.ipynb)
# Logos are organized in subdirectories: logos/PartyName/*.jpg
LOGO_ROOT = "/kaggle/input/shared-task-political-party-logos/logos"

logo_embs = []
logo_party = []

skipped = 0
loaded = 0

print(f"Loading logos from: {LOGO_ROOT}")
print(f"Checking for subdirectories (one per party)...")

# Iterate through party subdirectories
for party in sorted(os.listdir(LOGO_ROOT)):
    party_dir = os.path.join(LOGO_ROOT, party)
    if not os.path.isdir(party_dir):
        continue
        
    print(f"  → {party}:", end=" ")
    party_count = 0
    
    # Load all image files from this party's directory
    for f in os.listdir(party_dir):
        # Skip SVGs (PIL can't load them)
        if not f.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".avif")):
            continue
        
        img_path = os.path.join(party_dir, f)
        try:
            # Use the same embed_image function we defined earlier
            emb = embed_image(img_path)
            logo_embs.append(emb)
            logo_party.append(party)
            loaded += 1
            party_count += 1
        except Exception as e:
            skipped += 1
            print(f"\n     ⚠️  Failed {f}: {type(e).__name__}")
    
    print(f"{party_count} logos")

print(f"\n{'='*60}")
print(f"✅ Loaded logos: {loaded}")
print(f"   Skipped logos: {skipped}")

In [ ]:
# Safety check and convert to numpy arrays
if loaded == 0:
    print("❌ ERROR: No logo embeddings were created!")
    print(f"   Tried path: {LOGO_ROOT}")
    print("   Your submission used logo features, so this MUST work!")
    print("\n   Troubleshooting:")
    print("   1. Check if LOGO_ROOT path exists")
    print("   2. Check if it contains subdirectories (one per party)")
    print("   3. Check if subdirectories contain .jpg/.png files")
else:
    logo_embs = np.vstack(logo_embs)
    logo_party = np.array(logo_party)
    print(f"   Final logo embedding shape: {logo_embs.shape}")
    print(f"   Unique parties: {len(set(logo_party))}")
    print(f"   Parties: {sorted(set(logo_party))}")
    print(f"{'='*60}")

In [ ]:
# STEP 3 — Compute logo similarity for EACH meme (exactly as in st2-multimodal.ipynb)
# Returns 3 features: [max_sim, top2_gap, hits]

def logo_features(img_emb, thresh=0.25):
    """
    Compute logo similarity features (matches st2-multimodal.ipynb exactly):
    - max_sim: Maximum cosine similarity to any logo
    - top2_gap: Difference between top-2 similarities
    - hits: Count of similarities above threshold (default 0.25)
    """
    sims = img_emb @ logo_embs.T
    top1 = sims.max()
    top2 = np.partition(sims, -2)[-2] if len(sims) > 1 else 0.0
    
    return [
        float(top1),
        float(top1 - top2),
        int((sims > thresh).sum())
    ]

print("\n" + "="*60)
print("Computing logo features for all memes...")
print("="*60)

logo_feat_list = []

for meme_emb in tqdm(img_embs, desc="Logo Features"):
    feats = logo_features(meme_emb, thresh=0.25)
    logo_feat_list.append(feats)

logo_feats = np.array(logo_feat_list, dtype=np.float32)
print(f"\n✅ Logo features shape: {logo_feats.shape}")
print(f"   Features: [max_sim, top2_gap, hits]")
print(f"   Sample values (first meme): {logo_feats[0]}")
print(f"   Min values: {logo_feats.min(axis=0)}")
print(f"   Max values: {logo_feats.max(axis=0)}")
print(f"   Mean values: {logo_feats.mean(axis=0)}")
print("="*60)

In [ ]:
# Combined features (CLIP + Face + Logo)
X = np.hstack([img_embs, face_feats, logo_feats])

print(f"\n{'='*60}")
print(f"FINAL FEATURE MATRIX (CLIP + Face + Logo)")
print(f"{'='*60}")
print(f"CLIP embeddings: {img_embs.shape[1]:>4}d")
print(f"  Face features: {face_feats.shape[1]:>4}d  (num_faces, max_face_area)")
print(f"  Logo features: {logo_feats.shape[1]:>4}d  (max_sim, top2_gap, hits)")
print(f"───────────────────────────")
print(f"          TOTAL: {X.shape[1]:>4}d")
print(f"        Samples: {X.shape[0]:>4}")
print(f"{'='*60}")

expected_shape = (1303, 773)
if X.shape == expected_shape:
    print(f"✅✅✅ PERFECT MATCH! Shape {X.shape}")
    print(f"     This matches your official submission!")
    print(f"     Ready to run CV evaluation and get correct results!")
else:
    print(f"⚠️⚠️⚠️  Shape mismatch!")
    print(f"     Expected: {expected_shape}")
    print(f"     Got:      {X.shape}")
    print(f"     Check feature extraction steps above!")

print(f"{'='*60}\n")

## 5. Ablation Study: Feature Contribution Analysis
**Testing three feature combinations to show what each component contributes:**
1. **CLIP only** (768d) - baseline
2. **CLIP + Face** (770d) - adds face detection
3. **CLIP + Face + Logo** (773d) - final submission (adds logo matching)

In [ ]:
# ABLATION STUDY: Test three feature combinations
def ablation_study():
    """
    Compare three feature combinations:
    1. CLIP only (768d)
    2. CLIP + Face (770d) 
    3. CLIP + Face + Logo (773d)
    """
    from sklearn.svm import LinearSVC
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import f1_score
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Prepare three feature matrices
    X_clip = clip_feats  # (1303, 768)
    X_clip_face = np.hstack([clip_feats, face_feats])  # (1303, 770)
    X_clip_face_logo = X  # (1303, 773) - full features
    
    feature_combinations = [
        ("CLIP only", X_clip, 768),
        ("CLIP + Face", X_clip_face, 770),
        ("CLIP + Face + Logo", X_clip_face_logo, 773)
    ]
    
    results = []
    
    print("=" * 80)
    print("ABLATION STUDY: Testing Feature Combinations")
    print("=" * 80)
    
    for feat_name, X_feat, expected_dim in feature_combinations:
        print(f"\n{'─' * 80}")
        print(f"Testing: {feat_name} ({X_feat.shape[1]}d)")
        print(f"{'─' * 80}")
        
        for lang in ['tamil', 'malayalam']:
            mask = (df["lang"].values == lang)
            X_lang = X_feat[mask]
            y_stance = df.loc[mask, "stance_internal"].values
            y_target = df.loc[mask, "target_internal"].values
            
            # Stance F1
            clf_stance = LinearSVC(class_weight="balanced", max_iter=20000, random_state=42)
            pred_stance = cross_val_predict(clf_stance, X_lang, y_stance, cv=skf)
            stance_f1 = f1_score(y_stance, pred_stance, average='macro')
            
            # Target F1
            clf_target = LinearSVC(class_weight="balanced", max_iter=20000, random_state=42)
            pred_target = cross_val_predict(clf_target, X_lang, y_target, cv=skf)
            target_f1 = f1_score(y_target, pred_target, average='macro')
            
            avg_f1 = (stance_f1 + target_f1) / 2
            
            print(f"  {lang.title():>10s}: Stance={stance_f1:.4f}  Target={target_f1:.4f}  Avg={avg_f1:.4f}")
            
            results.append({
                'Features': feat_name,
                'Dimensions': expected_dim,
                'Language': lang.title(),
                'Stance F1': stance_f1,
                'Target F1': target_f1,
                'Average F1': avg_f1
            })
    
    results_df = pd.DataFrame(results)
    
    # Print summary table
    print("\n" + "=" * 80)
    print("ABLATION STUDY RESULTS TABLE")
    print("=" * 80)
    
    # Pivot table for paper
    pivot_df = results_df.pivot_table(
        index=['Features', 'Dimensions'],
        columns='Language',
        values=['Stance F1', 'Target F1', 'Average F1']
    )
    
    print("\nFormatted Table:")
    print(pivot_df.to_string())
    
    # Alternative: Row-based format (better for paper)
    print("\n" + "=" * 80)
    print("TABLE FOR PAPER (Copy this format):")
    print("=" * 80)
    
    for feat_name in ["CLIP only", "CLIP + Face", "CLIP + Face + Logo"]:
        feat_rows = results_df[results_df['Features'] == feat_name]
        tamil_row = feat_rows[feat_rows['Language'] == 'Tamil'].iloc[0]
        malay_row = feat_rows[feat_rows['Language'] == 'Malayalam'].iloc[0]
        
        print(f"\n{feat_name} ({feat_rows.iloc[0]['Dimensions']}d):")
        print(f"  Tamil:     Stance={tamil_row['Stance F1']:.4f}  Target={tamil_row['Target F1']:.4f}  Avg={tamil_row['Average F1']:.4f}")
        print(f"  Malayalam: Stance={malay_row['Stance F1']:.4f}  Target={malay_row['Target F1']:.4f}  Avg={malay_row['Average F1']:.4f}")
    
    # Calculate improvements
    print("\n" + "=" * 80)
    print("IMPROVEMENT ANALYSIS:")
    print("=" * 80)
    
    for lang in ['Tamil', 'Malayalam']:
        lang_data = results_df[results_df['Language'] == lang]
        clip_only = lang_data[lang_data['Features'] == 'CLIP only'].iloc[0]
        clip_face = lang_data[lang_data['Features'] == 'CLIP + Face'].iloc[0]
        clip_face_logo = lang_data[lang_data['Features'] == 'CLIP + Face + Logo'].iloc[0]
        
        print(f"\n{lang}:")
        print(f"  Face adds:  Stance +{(clip_face['Stance F1'] - clip_only['Stance F1']):.4f}  Target +{(clip_face['Target F1'] - clip_only['Target F1']):.4f}")
        print(f"  Logo adds:  Stance +{(clip_face_logo['Stance F1'] - clip_face['Stance F1']):.4f}  Target +{(clip_face_logo['Target F1'] - clip_face['Target F1']):.4f}")
        print(f"  Total gain: Stance +{(clip_face_logo['Stance F1'] - clip_only['Stance F1']):.4f}  Target +{(clip_face_logo['Target F1'] - clip_only['Target F1']):.4f}")
    
    print("\n" + "=" * 80 + "\n")
    
    return results_df

# Run ablation study (will take ~90 minutes: 3 features × 2 languages × 2 tasks × 5 folds)
print("⚠️  This will take approximately 90 minutes to complete (~30 min per feature set)")
print("    Running 5-fold CV for 3 feature combinations × 2 languages × 2 tasks = 60 models\n")

ablation_results = ablation_study()

## 6. Cross-Validation Results (Per Language)
**NOTE:** Section 5 above shows ablation study. This section just re-confirms the final CLIP+Face+Logo results.

In [ ]:
# TABLE 2: Cross-validation results
def cv_evaluation():
    results = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for lang in ['tamil', 'malayalam']:
        mask = (df["lang"].values == lang)
        X_lang = X[mask]
        y_stance = df.loc[mask, "stance_internal"].values
        y_target = df.loc[mask, "target_internal"].values
        
        # Stance F1
        clf_stance = LinearSVC(class_weight="balanced", max_iter=20000)
        pred_stance = cross_val_predict(clf_stance, X_lang, y_stance, cv=skf)
        stance_f1 = f1_score(y_stance, pred_stance, average='macro')
        
        # Target F1
        clf_target = LinearSVC(class_weight="balanced", max_iter=20000)
        pred_target = cross_val_predict(clf_target, X_lang, y_target, cv=skf)
        target_f1 = f1_score(y_target, pred_target, average='macro')
        
        results.append({
            'Language': lang.title(),
            'Level 1 (Stance) F1': f"{stance_f1:.4f}",
            'Level 2 (Target) F1': f"{target_f1:.4f}",
            'Average F1': f"{(stance_f1 + target_f1) / 2:.4f}"
        })
    
    results_df = pd.DataFrame(results)
    print("\n=== TABLE 2: Cross-Validation Results (5-fold) ===")
    print(results_df.to_string(index=False))
    print("\nLaTeX format:")
    print(results_df.to_latex(index=False))
    
    return results_df

cv_results_df = cv_evaluation()

## 7. Official Test Results Comparison

In [ ]:
# TABLE 3: Official vs CV Results
official_results = pd.DataFrame([
    {
        'Language': 'Malayalam',
        'Split': 'Official Test',
        'Level 1 F1': 0.9638,
        'Level 2 F1': 0.6222,
        'Avg F1': 0.7930,
        'Rank': 1
    },
    {
        'Language': 'Tamil',
        'Split': 'Official Test',
        'Level 1 F1': 0.9271,
        'Level 2 F1': 0.6060,
        'Avg F1': 0.7666,
        'Rank': 6
    }
])

print("\n=== TABLE 3: Official Test Results ===")
print(official_results.to_string(index=False))
print("\nLaTeX format:")
print(official_results.to_latex(index=False))

## 8. Confusion Matrices

In [ ]:
# Generate confusion matrices for both tasks
def plot_confusion_matrices():
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    tasks = ['stance_internal', 'target_internal']
    task_names = ['Stance (Level 1)', 'Target (Level 2)']
    
    for lang_idx, lang in enumerate(['tamil', 'malayalam']):
        mask = (df["lang"].values == lang)
        X_lang = X[mask]
        
        for task_idx, (task, task_name) in enumerate(zip(tasks, task_names)):
            y = df.loc[mask, task].values
            
            clf = LinearSVC(class_weight="balanced", max_iter=20000)
            pred = cross_val_predict(clf, X_lang, y, cv=skf)
            
            cm = confusion_matrix(y, pred)
            labels = sorted(set(y))
            
            ax = axes[task_idx, lang_idx]
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                       xticklabels=labels, yticklabels=labels, ax=ax)
            ax.set_title(f"{lang.title()} - {task_name}")
            ax.set_ylabel('True Label')
            ax.set_xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confusion_matrices.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {OUTPUT_DIR}/confusion_matrices.png")

plot_confusion_matrices()

## 9. Feature Analysis

In [ ]:
# Figure: Face detection correlation with target class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, lang in enumerate(['tamil', 'malayalam']):
    lang_df = df[df['lang'] == lang]
    
    # Group by target class
    face_by_target = lang_df.groupby('target_internal')['num_faces'].mean()
    
    ax = axes[idx]
    face_by_target.plot(kind='bar', ax=ax, color='skyblue')
    ax.set_title(f"{lang.title()} - Avg Faces by Target")
    ax.set_xlabel('Target Class')
    ax.set_ylabel('Average Number of Faces')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/face_analysis.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/face_analysis.png")

## 10. Performance Comparison Chart

In [ ]:
# Figure: CV vs Official Test comparison
# You'll need to manually add CV results from your TABLE 2
comparison_data = pd.DataFrame([
    {'Language': 'Malayalam', 'Dataset': 'CV (5-fold)', 'Level 1': 0.65, 'Level 2': 0.48},  # UPDATE WITH YOUR CV RESULTS
    {'Language': 'Malayalam', 'Dataset': 'Official Test', 'Level 1': 0.9638, 'Level 2': 0.6222},
    {'Language': 'Tamil', 'Dataset': 'CV (5-fold)', 'Level 1': 0.80, 'Level 2': 0.58},  # UPDATE WITH YOUR CV RESULTS
    {'Language': 'Tamil', 'Dataset': 'Official Test', 'Level 1': 0.9271, 'Level 2': 0.6060},
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, level in enumerate(['Level 1', 'Level 2']):
    ax = axes[idx]
    
    # Reshape for grouped bar chart
    for lang in ['Malayalam', 'Tamil']:
        lang_data = comparison_data[comparison_data['Language'] == lang]
        x_pos = [0, 1] if lang == 'Malayalam' else [2.5, 3.5]
        ax.bar(x_pos, lang_data[level].values, width=0.8, 
               label=lang, alpha=0.8)
    
    ax.set_title(f"{level} F1-Score")
    ax.set_ylabel('F1-Score')
    ax.set_xticks([0.5, 3])
    ax.set_xticklabels(['Malayalam', 'Tamil'])
    ax.legend(['CV', 'Test'], loc='lower right')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cv_vs_test.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/cv_vs_test.png")

## 11. Per-Class F1 Scores

In [ ]:
# Detailed per-class analysis
def per_class_analysis():
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for lang in ['tamil', 'malayalam']:
        print(f"\n{'='*60}")
        print(f"{lang.upper()} - Per-Class Performance")
        print('='*60)
        
        mask = (df["lang"].values == lang)
        X_lang = X[mask]
        
        for task in ['stance_internal', 'target_internal']:
            y = df.loc[mask, task].values
            
            clf = LinearSVC(class_weight="balanced", max_iter=20000)
            pred = cross_val_predict(clf, X_lang, y, cv=skf)
            
            print(f"\n{task.replace('_internal', '').upper()}:")
            print(classification_report(y, pred, digits=4))

per_class_analysis()

## 11. Sample Meme Examples (for paper)

In [ ]:
# Select representative examples
def show_sample_memes():
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    
    # Select one example from each category
    examples = [
        ('tamil', 'SUPPORT', 'PERSON'),
        ('tamil', 'TROLL', 'PARTY'),
        ('malayalam', 'SUPPORT', 'PARTY'),
        ('malayalam', 'TROLL', 'INTERSECTION'),
    ]
    
    for idx, (lang, stance, target) in enumerate(examples):
        # Find a sample
        sample = df[(df['lang'] == lang) & 
                   (df['stance_internal'] == stance) & 
                   (df['target_internal'] == target)].iloc[0]
        
        img = Image.open(sample['image_path'])
        
        ax = axes[idx // 2, idx % 2]
        ax.imshow(img)
        ax.set_title(f"{lang.title()}\n{stance} - {target}", fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/sample_memes.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {OUTPUT_DIR}/sample_memes.png")
    print("⚠️  IMPORTANT: Blur or anonymize faces before using in paper!")

show_sample_memes()

## 12. System Architecture Diagram (TODO: Create manually)

In [ ]:
print("""
✅ All figures generated in: paper_figures/

📊 Generated Files:
  1. confusion_matrices.png - Confusion matrices for all tasks
  2. face_analysis.png - Face detection statistics
  3. cv_vs_test.png - CV vs Test performance
  4. sample_memes.png - Example memes (⚠️ anonymize before use!)

📝 TODO - Create manually:
  5. System architecture diagram (use draw.io, PowerPoint, or TikZ)
     - Show: Input Image → CLIP → [768d] 
                        → Face Detection → [2d]
                        → Logo Matching → [3d]
                        → Concatenate → [773d]
                        → LinearSVC → Predictions

💡 Use these figures in your paper!
""")